# Model 2: Clustering Setup

- `breed_clustering_full.csv`: individual AKC trait scores plus comparison labels.
- `breed_clustering_avg.csv`: AKC trait-group avg scores plus comparison labels.

In [9]:
from pathlib import Path
import sys

for base_dir in [Path.cwd(), *Path.cwd().parents]:
    if (base_dir / "models").exists():
        sys.path.insert(0, str(base_dir))
        break

from models.setup_utils import (
    build_avg_traits,
    data_dirs,
    find_project_root,
    load_interim_data,
    prepare_numeric_traits,
    add_popularity_tier,
    write_clustering_csv,
)

PROJECT_ROOT = find_project_root()
INTERIM_DIR, PROCESSED_DIR = data_dirs(PROJECT_ROOT)

## Load Data

In [10]:
breed_traits, breed_ranks, breed_groups = load_interim_data(INTERIM_DIR, include_groups=True)
breed_ranks = add_popularity_tier(breed_ranks)
breed_traits, numeric_trait_cols = prepare_numeric_traits(breed_traits)

comparison_labels = breed_ranks[["Breed", "Average Rank", "Popularity Tier"]].merge(
    breed_groups.rename(columns={"Group": "AKC Group"}),
    on="Breed",
    how="inner",
)

## Full Trait CSV

In [11]:
breed_clustering_full = breed_traits[["Breed", *numeric_trait_cols]].merge(
    comparison_labels,
    on="Breed",
    how="inner",
)

full_output_path, breed_clustering_full_course = write_clustering_csv(
    breed_clustering_full,
    numeric_trait_cols,
    PROCESSED_DIR / "breed_clustering_full.csv",
)

## Grouped Avg Trait CSV

In [12]:
breed_clustering_avg, avg_trait_cols = build_avg_traits(breed_traits)
breed_clustering_avg = breed_clustering_avg.merge(
    comparison_labels,
    on="Breed",
    how="inner",
)

avg_output_path, breed_clustering_avg_course = write_clustering_csv(
    breed_clustering_avg,
    avg_trait_cols,
    PROCESSED_DIR / "breed_clustering_avg.csv",
)